# Clarkson II hold-time threshold visualization

This notebook visualizes the sensitivity-analysis CSV files so the long-hold problem can be understood without reading large tables manually.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)
DATA_DIR = PROJECT_ROOT / 'clarkson2_preprocessing' / 'outputs' / 'hold_threshold_sensitivity'
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

global_df = pd.read_csv(DATA_DIR / 'hold_threshold_sensitivity.csv')
user_df = pd.read_csv(DATA_DIR / 'hold_threshold_sensitivity_by_user.csv')
hist_df = pd.read_csv(DATA_DIR / 'hold_time_histogram.csv')
quantile_df = pd.read_csv(DATA_DIR / 'hold_distribution_quantiles.csv')

plt.style.use('seaborn-v0_8-whitegrid')
print('Global rows:', len(global_df))
print('User-threshold rows:', len(user_df))

Global rows: 3
User-threshold rows: 300


## 1. Retained versus removed pairs

This plot shows why increasing the threshold from 5 to 30 seconds does not solve the problem.

In [2]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(global_df))
width = 0.62
ax.bar(x, global_df['kept_percent'], width, label='Kept', color='#2a9d8f')
ax.bar(x, global_df['removed_percent'], width, bottom=global_df['kept_percent'],
       label='Removed: hold time above threshold', color='#e76f51')
ax.set_xticks(x, [f'{value:g}s' for value in global_df['threshold_seconds']])
ax.set_ylim(0, 100)
ax.set_xlabel('Hold-time threshold')
ax.set_ylabel('Percentage of nonnegative pairs')
ax.set_title('Effect of hold-time threshold on retained pairs')
ax.legend()
for index, row in global_df.iterrows():
    ax.text(index, row['kept_percent'] / 2, f"{row['kept_percent']:.1f}%", ha='center', va='center', color='white', fontweight='bold')
    ax.text(index, row['kept_percent'] + row['removed_percent'] / 2, f"{row['removed_percent']:.1f}%", ha='center', va='center', color='white', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_threshold_retention.png', dpi=180)
plt.close(fig)

## 2. Hold-time distribution

The x-axis uses logarithmic spacing because the distribution contains both ordinary key holds and extremely large values. The final bin contains all holds above 300 seconds.

In [3]:
fig, ax = plt.subplots(figsize=(10, 5.5))
labels = hist_df['hold_time_bin_seconds'].tolist()
counts = hist_df['count'].tolist()
ax.bar(np.arange(len(labels)), counts, color='#457b9d')
ax.set_yscale('log')
ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha='right')
ax.set_xlabel('Hold-time bin in seconds')
ax.set_ylabel('Pair count, logarithmic scale')
ax.set_title('Hold-time distribution after timestamp-reset repair')
ax.text(0.98, 0.95, f"{counts[-1]:,} pairs exceed 300 seconds", transform=ax.transAxes,
        ha='right', va='top', bbox=dict(facecolor='white', alpha=0.85))
fig.tight_layout()
fig.savefig(FIG_DIR / '02_hold_time_distribution_log.png', dpi=180)
plt.close(fig)

## 3. Approximate hold-time quantiles

These quantiles were estimated from a fixed-size reservoir sample during the scan. They show where the distribution becomes implausible.

In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(quantile_df['quantile'] * 100, quantile_df['hold_seconds_approx'], marker='o', color='#264653')
ax.set_yscale('log')
ax.set_xlabel('Quantile (%)')
ax.set_ylabel('Approximate hold time in seconds, logarithmic scale')
ax.set_title('Approximate hold-time quantiles')
for _, row in quantile_df.iterrows():
    ax.annotate(f"{row['hold_seconds_approx']:.2g}s", (row['quantile'] * 100, row['hold_seconds_approx']),
                textcoords='offset points', xytext=(4, 5), fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / '03_hold_time_quantiles.png', dpi=180)
plt.close(fig)

## 4. Per-user retention heatmap

This reveals whether the long-hold problem is isolated to a few users or widespread.

In [5]:
retention = user_df.pivot(index='user_id', columns='threshold_seconds', values='kept_percent')
retention = retention.sort_values(by=5.0)
fig, ax = plt.subplots(figsize=(8, max(8, len(retention) * 0.16)))
image = ax.imshow(retention.to_numpy(), aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(np.arange(len(retention.columns)), [f'{value:g}s' for value in retention.columns])
step = max(1, len(retention) // 25)
shown = np.arange(0, len(retention), step)
ax.set_yticks(shown, retention.index.to_numpy()[shown])
ax.set_xlabel('Hold-time threshold')
ax.set_ylabel('User ID, sorted by 5-second retention')
ax.set_title('Per-user percentage of pairs retained')
fig.colorbar(image, ax=ax, label='Retained pairs (%)')
fig.tight_layout()
fig.savefig(FIG_DIR / '04_per_user_retention_heatmap.png', dpi=180)
plt.close(fig)

## 5. User-level 5-second retention

The lowest-retention users are the most important cases for investigating pairing and logger behavior.

In [6]:
five_second = user_df[user_df['threshold_seconds'] == 5.0].sort_values('kept_percent')
worst = five_second.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(worst['user_id'].astype(str), worst['kept_percent'], color='#e76f51')
ax.set_xlim(0, 100)
ax.set_xlabel('Pairs retained at 5-second threshold (%)')
ax.set_ylabel('User ID')
ax.set_title('20 users most affected by the 5-second threshold')
for position, value in enumerate(worst['kept_percent']):
    ax.text(value + 1, position, f'{value:.1f}%', va='center', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / '05_worst_user_retention.png', dpi=180)
plt.close(fig)

worst.to_csv(FIG_DIR / '05_worst_user_retention.csv', index=False)

## 6. Pair count versus retention

This checks whether low retention is mainly a problem of users with very little data or also affects users with large streams.

In [7]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(five_second['nonnegative_pairs'], five_second['kept_percent'], alpha=0.7, color='#2a9d8f')
ax.set_xscale('log')
ax.set_xlabel('Nonnegative paired events per user, logarithmic scale')
ax.set_ylabel('Pairs retained at 5 seconds (%)')
ax.set_title('User data volume versus 5-second retention')
for _, row in five_second.nsmallest(5, 'kept_percent').iterrows():
    ax.annotate(str(row['user_id']), (row['nonnegative_pairs'], row['kept_percent']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / '06_volume_vs_retention.png', dpi=180)
plt.close(fig)

## Generated figures

The PNG files are saved in the `figures` subfolder. The most important plots for the project report are the threshold-retention chart, hold-time distribution, and per-user retention heatmap.